In [1]:
import json

In [2]:
import pickle

In [3]:
import tqdm

In [4]:
from dotenv import load_dotenv

In [5]:
load_dotenv()

True

In [6]:
import pandas as pd

In [7]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [8]:
MODEL_ID = "google/gemma-4-E2B-it"

In [9]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [11]:
df = pd.read_excel("data.xlsx", header=0)

In [12]:
df.head()

,title_rus,title_eng,goals,tasks,annotation,description,expectations,product_result,result_criterias,social_effect,commercial_effect
0,Исследование приоритетов и механизмов реализац...,Study of Priorities and Mechanisms for Impleme...,Целью проекта является эмпирическая проверка г...,1) Анализ повестки международных доноров\n2) А...,Работа международных фондов (доноров) должна п...,"Согласно определению международных фондов, про...",Аналитический отчет по избранным странам.\n\nС...,NaN,NaN,NaN,NaN
1,Антрополе - научно-популярный видео-подкаст о ...,Anthropole is a Popular Science Video Podcast ...,Выпустить и популяризовать 27 видео-подкастов ...,Снять и смонтировать подкасты;\nРазработать ст...,"\tАнтрополе - научно-популярный проект, в рамк...","Социальное знание близко и интересно обществу,...","Студенты получат опыт монтажа, продвижения и к...",Регулярный видео-подкаст (+экспедиционные филь...,Опубликованы 27 видео-подкастов о необычных со...,Популяризация социального знания должна привес...,Планируется в течение года выйти на окупаемост...
2,"Разработка, создание и ведение сайта, посвящен...","Design, Development and Implementation of a We...","Для того, чтобы получить более полное представ...",Подготовка технического задания для разработчи...,Художественное образование и творчество художн...,Тема обучения арабских художников в художестве...,"Навыки создания сайта (полный цикл, от подгото...","Создание сайта, посвященного истории арабских ...",Выполнение заданий руководителя проекта,Укрепление международных связей с художниками ...,NaN
3,Перевод с английского языка коллективной моног...,Translation from English of the collective mon...,Результатом проекта станет качественный перево...,Результатом проекта станет качественный перево...,"Коллективная монография, авторы которой являют...","Коллективная монография, авторы которой являют...",Участники проекта приобретут новые знания в об...,Результатом проекта станет качественный перево...,Критериями достижения результата будет возможн...,NaN,NaN
4,Сеть военно-политических союзов в Евразии: баз...,Network of Military in Eurasia: a Database,Целью проекта является создание базы данных со...,1.\tОпределение методики включения союзов в ба...,Проект посвящен изучению сети военно-политичес...,Проект посвящен анализу истории существования ...,1.\tОбучение навыкам сбора и анализа данных 2....,База данных союзов 1815-2024 в Евразии.,Создание базы данных как минимум тысячи диадны...,NaN,NaN


In [13]:
df = df.fillna("")

In [14]:
df.drop_duplicates(inplace=True)

In [15]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can extract short and concise tags for student projects title in russian.
Please, for a given student project title return a list of corresponding tags, which depicts the field of science to which the project is relevant.
For example, for a project title "Рекомендательная система для студенческих проектов" you answer might be the following list of strings with underscores.
["machine_learning", "recommender_systems", "software_engineering"]
Also another example, for a project title "Прогнозирование эффектов процентной политики Банка России" you answer might be the following list of strings with underscores.
["economics", "macroeconomics", "monetary_policy", "forecasting", "time_series"]
Return strictly a list of strings as an answer.
"""

In [16]:
messages = []

In [17]:
for row in df.iterrows():
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row[1]["title_rus"]}
        )
    )

In [18]:
tags_set = set()

In [19]:
titles_with_tags_dict = dict()

In [20]:
for message in tqdm.tqdm(messages):
    text = processor.apply_chat_template(
        message, 
        tokenize=False, 
        add_generation_prompt=True, 
        enable_thinking=False
    )
    
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    
    outputs = model.generate(**inputs, max_new_tokens=1024)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    answer = json.loads(processor.parse_response(response)["content"])
    
    titles_with_tags_dict[message[1]["content"]] = answer
    tags_set = tags_set.union(answer)

100%|██████████| 1250/1250 [53:01<00:00,  2.55s/it]


In [21]:
with open("titles_with_tags_dict.pkl", "wb") as f:
    pickle.dump(titles_with_tags_dict, f)

In [22]:
with open("tags_set.pkl", "wb") as f:
    pickle.dump(tags_set, f)

In [23]:
len(tags_set)

1914

In [25]:
list(titles_with_tags_dict.values())[0]

['international_relations',
 'political_economics',
 'policy_analysis',
 'brics',
 'geopolitics',
 'international_policy',
 'comparative_politics']

In [26]:
list(titles_with_tags_dict.values())[-1]

['regional_studies', 'methodology', 'data_collection', 'geography']